# 01 - SMILES ve RDKit Mol Nesnesi

Bu notebookta RDKit'te molekülle çalışmanın ilk teknik adımını öğreneceğiz:

**SMILES metni → RDKit `Mol` nesnesi → temel molekül kontrolü → canonical SMILES → basit görselleştirme**

Bu çalışma eğitim amaçlıdır. Gerçek veri indirme, final dataset seçimi, modelleme, train/test split, metric/evaluation, Macro F1, benchmark veya bilimsel sonuç üretimi yapılmaz.


## Bu notebookta ne öğrenilecek?

Bu notebookun sonunda şunu rahatça söyleyebilmelisin:

> RDKit'te molekülle çalışmanın başlangıç noktası çoğu zaman SMILES metnini `Mol` nesnesine çevirmektir. `Mol` nesnesi yalnızca bir string değildir; atom, bağ, halka ve kimyasal yapı bilgisini taşıyan hesaplanabilir bir temsildir.

Öğreneceğimiz ana kavramlar:

- SMILES nedir?
- RDKit `Mol` nesnesi nedir?
- Geçerli ve geçersiz SMILES farkı nasıl görülür?
- Canonical SMILES neden duplicate farkındalığı için yararlıdır?
- Basit molekül bilgileri nasıl okunur?
- Molekül çizimi kalite kontrol düşüncesine nasıl yardım eder?


## Bu notebookta ne yapılmayacak?

- Gerçek veri indirilmeyecek.
- `data/raw/` veya `data/processed/` kullanılmayacak.
- `nmrshiftdb2rawdata.nmredata.sd` dosyası ana veri olarak okunmayacak.
- Model kurulmayacak.
- Train/test split yapılmayacak.
- Metric, benchmark, skor veya sahte sonuç üretilmeyecek.
- Final dataset seçimi, kaynak seçimi, ADR kabulü veya learning gate geçişi yapılmayacak.


## Önceki notebookla bağlantı

`00_rdkit_projedeki_rolu.ipynb` notebookunda RDKit'in projedeki genel rolünü gördük: RDKit, molekül yapısını Python içinde temsil etmeye ve kimyasal veriyi daha düzenli düşünmeye yardım eder.

Bu notebookta artık ilk teknik temsile geçiyoruz. Bir molekülü metin olarak yazacağız, RDKit'e okutacağız ve RDKit'in bize verdiği `Mol` nesnesinin ne taşıdığını anlamaya başlayacağız.


## SMILES nedir?

SMILES (Simplified Molecular Input Line Entry System), molekül yapısını tek satırlık metin olarak yazmanın yaygın bir yoludur.

SMILES bir insan çizimi değildir. Yani kağıttaki yapı formülünün aynısı gibi düşünülmemelidir. Daha çok bilgisayarın okuyabileceği kompakt bir yapı temsilidir.

Örnekler:

- Ethanol: `CCO`
- Acetone: `CC(=O)C`
- Acetic acid: `CC(=O)O`
- Benzene: `c1ccccc1`
- Ethyl acetate: `CCOC(=O)C`
- Aniline: `Nc1ccccc1`

Bu yazımlar kısa görünür, ama doğru yorumlandığında atom ve bağ ilişkilerini temsil eder.


## RDKit Mol nesnesi nedir?

RDKit'te temel dönüşüm çoğu zaman şudur:

```python
mol = Chem.MolFromSmiles("CCO")
```

`Chem.MolFromSmiles()` SMILES metnini okumaya çalışır. Başarılı olursa bir `Mol` nesnesi döndürür. Başarısız olursa genellikle `None` döndürür.

`Mol` nesnesi düz bir string değildir. İçinde atomlar, bağlar, halka bilgisi, aromatiklik gibi kimyasal yapı bilgisini taşıyan hesaplanabilir bir temsil vardır.


In [ ]:
# RDKit bu ortamda kurulu olmayabilir.
# Güvenli import kullanıyoruz: RDKit yoksa notebook tamamen çökmesin.

RDKit_AVAILABLE = False
rdkit_import_error = None
Draw = None

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, rdMolDescriptors
    try:
        from rdkit.Chem import Draw
    except Exception:
        Draw = None
    RDKit_AVAILABLE = True
except Exception as exc:
    rdkit_import_error = exc

if RDKit_AVAILABLE:
    print("RDKit bulundu. Teknik hücreler çalıştırılabilir.")
else:
    print("RDKit bu ortamda bulunamadı.")
    print("Notebook kavramsal olarak okunabilir; RDKit gerektiren hücreler güvenli şekilde atlanacak.")
    print(f"Import hatası: {type(rdkit_import_error).__name__}: {rdkit_import_error}")


## Oyuncak molekül listesi

İlk teknik notebooklarda küçük ve kolay kontrol edilebilir moleküllerle çalışıyoruz. Bu liste final proje dataset'i değildir; sadece SMILES ve `Mol` nesnesi kavramlarını öğrenmek içindir.

NMReDATA learning sample dosyası (`nmrshiftdb2rawdata.nmredata.sd`) bu notebookta ana veri olarak kullanılmaz. O dosya daha ileride, SDF/NMReDATA okuma ve debugging öğrenimi için `learning sample / pilot example` statüsünde devreye girebilir.


In [ ]:
toy_molecules = [
    {"molecule_name": "ethanol", "smiles": "CCO", "expected_note": "basit alkol"},
    {"molecule_name": "acetone", "smiles": "CC(=O)C", "expected_note": "basit keton/karbonil"},
    {"molecule_name": "acetic_acid", "smiles": "CC(=O)O", "expected_note": "karboksilik asit"},
    {"molecule_name": "benzene", "smiles": "c1ccccc1", "expected_note": "aromatik halka"},
    {"molecule_name": "ethyl_acetate", "smiles": "CCOC(=O)C", "expected_note": "ester örneği"},
    {"molecule_name": "aniline", "smiles": "Nc1ccccc1", "expected_note": "aromatik amin"},
    {"molecule_name": "cyclohexane", "smiles": "C1CCCCC1", "expected_note": "alifatik halka"},
    {"molecule_name": "invalid_broken_ring", "smiles": "C1CC", "expected_note": "bilerek geçersiz SMILES"},
]

def print_rows(rows, columns):
    """Küçük öğretici tabloları ek paket kullanmadan yazdırır."""
    widths = []
    for column in columns:
        values = [str(row.get(column, "")) for row in rows]
        widths.append(max([len(column)] + [len(value) for value in values]))

    header = " | ".join(column.ljust(width) for column, width in zip(columns, widths))
    line = "-+-".join("-" * width for width in widths)
    print(header)
    print(line)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(width) for column, width in zip(columns, widths)))

print_rows(toy_molecules, ["molecule_name", "smiles", "expected_note"])


## SMILES → Mol dönüşümü

Şimdi her SMILES metnini RDKit'e okutmayı deniyoruz. Geçerli olanlar `Mol` nesnesine dönüşür. Geçersiz olanlar `None` dönebilir.

`None` sonucu önemlidir: RDKit bu metni güvenilir bir molekül yapısı olarak okuyamamıştır. Bu durum veri temizliği ve hata kontrolü açısından ileride çok önemli hale gelir.


In [ ]:
parsed_molecules = []

if RDKit_AVAILABLE:
    for item in toy_molecules:
        mol = Chem.MolFromSmiles(item["smiles"])
        parsed_molecules.append({**item, "mol": mol})

    rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        rows.append({
            "molecule_name": item["molecule_name"],
            "smiles": item["smiles"],
            "mol_created": mol is not None,
            "message": "okundu" if mol is not None else "None döndü: SMILES kontrol edilmeli",
        })

    print_rows(rows, ["molecule_name", "smiles", "mol_created", "message"])
else:
    parsed_molecules = [{**item, "mol": None} for item in toy_molecules]
    print("RDKit olmadığı için SMILES -> Mol dönüşümü çalıştırılmadı.")
    print("Kavramsal fikir: geçerli SMILES bir Mol nesnesine dönüşür; geçersiz SMILES None dönebilir.")


## Mol nesnesinin string olmadığını görmek

SMILES metni bir string'dir. `Mol` nesnesi ise RDKit'in hesaplanabilir molekül temsilidir.

Bir `Mol` nesnesinden atom sayısı, bağ sayısı veya canonical SMILES gibi bilgiler üretilebilir. Bu yüzden `Mol`, yalnızca metni saklayan bir kutu gibi değil, kimyasal yapıyı temsil eden bir nesne gibi düşünülmelidir.


In [ ]:
if RDKit_AVAILABLE:
    example = next(item for item in parsed_molecules if item["molecule_name"] == "ethanol")
    mol = example["mol"]

    print(f"SMILES değeri: {example['smiles']}")
    print(f"SMILES Python tipi: {type(example['smiles']).__name__}")
    print(f"Mol Python tipi: {type(mol).__name__}")
    print(f"Atom sayısı: {mol.GetNumAtoms()}")
    print(f"Bağ sayısı: {mol.GetNumBonds()}")
else:
    print("RDKit olmadığı için Mol nesnesi tipi gösterilmedi.")
    print("Ana fikir: SMILES string, Mol ise atom ve bağ bilgisi taşıyan RDKit nesnesidir.")


## Canonical SMILES

Aynı molekül farklı SMILES yazımlarıyla ifade edilebilir. RDKit, molekülü okuduktan sonra canonical SMILES üretebilir. Bu, aynı yapının farklı metin yazımlarıyla tekrar tekrar görünmesini fark etmek için yararlı bir ilk kontroldür.

Ancak canonical SMILES tek başına bilimsel veri temizliği anlamına gelmez. Duplicate, stereokimya, tuzlar, tautomerler, ölçüm koşulları ve metadata gibi konular ayrıca düşünülmelidir.


In [ ]:
canonical_examples = [
    {"molecule_group": "ethanol", "input_smiles": "CCO"},
    {"molecule_group": "ethanol", "input_smiles": "OCC"},
    {"molecule_group": "ethanol", "input_smiles": "C(C)O"},
    {"molecule_group": "acetic_acid", "input_smiles": "CC(=O)O"},
    {"molecule_group": "acetic_acid", "input_smiles": "CC(O)=O"},
]

if RDKit_AVAILABLE:
    rows = []
    for item in canonical_examples:
        mol = Chem.MolFromSmiles(item["input_smiles"])
        rows.append({
            "molecule_group": item["molecule_group"],
            "input_smiles": item["input_smiles"],
            "canonical_smiles": Chem.MolToSmiles(mol) if mol is not None else "OKUNAMADI",
        })

    print_rows(rows, ["molecule_group", "input_smiles", "canonical_smiles"])
else:
    print("RDKit olmadığı için canonical SMILES örneği çalıştırılmadı.")
    print("Kavramsal fikir: farklı yazımlar, RDKit tarafından aynı canonical forma indirgenebilir.")


## Temel molekül bilgileri

Bir `Mol` nesnesinden basit bilgiler okunabilir:

- Atom sayısı
- Bağ sayısı
- Ağır atom sayısı
- Molekül formülü
- Molekül ağırlığı

Bunlar descriptor kavramına giriş niteliğindedir. Descriptor'lar ileride daha sistematik ele alınacaktır. Bu notebookta bu değerler yalnızca RDKit nesnesinin hesaplanabilir olduğunu göstermek için kullanılır.


In [ ]:
if RDKit_AVAILABLE:
    info_rows = []
    for item in parsed_molecules:
        mol = item["mol"]
        if mol is None:
            info_rows.append({
                "molecule_name": item["molecule_name"],
                "atom_count": "-",
                "bond_count": "-",
                "heavy_atom_count": "-",
                "formula": "-",
                "mol_weight": "-",
            })
            continue

        info_rows.append({
            "molecule_name": item["molecule_name"],
            "atom_count": mol.GetNumAtoms(),
            "bond_count": mol.GetNumBonds(),
            "heavy_atom_count": mol.GetNumHeavyAtoms(),
            "formula": rdMolDescriptors.CalcMolFormula(mol),
            "mol_weight": round(Descriptors.MolWt(mol), 2),
        })

    print_rows(info_rows, ["molecule_name", "atom_count", "bond_count", "heavy_atom_count", "formula", "mol_weight"])
else:
    print("RDKit olmadığı için temel molekül bilgileri hesaplanmadı.")


## Basit görselleştirme

RDKit molekül çizimi yapabilir. Görselleştirme özellikle kalite kontrol için yararlıdır: SMILES'ın beklediğimiz yapıya karşılık gelip gelmediğini hızlıca fark etmemizi sağlar.

Bir çizim, tek başına bilimsel doğrulama değildir. Ama yanlış SMILES, beklenmeyen halka, yanlış fonksiyonel grup veya okunamayan molekül gibi durumları fark etmeye yardım eder.


In [ ]:
if RDKit_AVAILABLE and Draw is not None:
    valid_items = [item for item in parsed_molecules if item["mol"] is not None]
    mols = [item["mol"] for item in valid_items]
    legends = [item["molecule_name"] for item in valid_items]

    # Notebook arayüzünde bu nesne molekül grid'i olarak görüntülenir.
    grid_image = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(220, 180), legends=legends)
    grid_image
elif RDKit_AVAILABLE:
    print("RDKit bulundu, ancak Draw modülü bu ortamda kullanılamadı.")
else:
    print("RDKit olmadığı için molekül çizimi yapılmadı.")
    print("Kavramsal fikir: çizim, SMILES'ın beklenen kimyasal yapıya karşılık gelip gelmediğini hızlıca kontrol etmeye yardım eder.")


## ¹³C NMR projesiyle bağlantı

Bu proje ¹³C NMR spektrumlarından fonksiyonel grup tahmini yapmayı hedefler. Böyle bir projede yapı bilgisi önemlidir çünkü fonksiyonel gruplar molekül yapısında tanımlanır.

SMILES ve `Mol` nesnesi ileride şu işler için temel oluşturur:

- SMARTS ile fonksiyonel grup veya alt yapı arama.
- Descriptor ve fingerprint üretmenin ne anlama geldiğini öğrenme.
- Molekül tekrarları ve duplicate riskleri hakkında düşünme.
- SDF/NMReDATA gibi daha gerçekçi dosyalarda molecule/property bağlantısını kalite kontrolle inceleme.

Bu notebookta modelleme yapılmadı. Burada yalnızca molekül temsilinin ilk adımı öğreniliyor.


## Mini alıştırmalar

Aşağıdaki alıştırmaları kendi yorumunla deneyebilirsin:

1. Oyuncak molekül listesine 3 yeni SMILES ekle.
2. Bilerek geçersiz bir SMILES yaz ve RDKit'in nasıl davrandığını yorumla.
3. Aynı molekül için iki farklı SMILES yazımı bul ve canonical SMILES çıktısını karşılaştır.
4. Bir molekülün atom sayısı ve bağ sayısını kimyasal yapısıyla ilişkilendirerek açıkla.

Bu alıştırmalar eğitim amaçlıdır. Veri kaynağı seçimi, modelleme veya bilimsel sonuç üretimi değildir.


## Kapanış Özeti

Bu notebookta SMILES'ın molekülü metin olarak temsil eden kompakt bir gösterim olduğunu gördük. RDKit'in `Chem.MolFromSmiles()` fonksiyonu ile SMILES metnini `Mol` nesnesine çevirmeye çalıştığını öğrendik. `Mol` nesnesinin yalnızca bir string olmadığını; atom, bağ ve yapı bilgisi taşıyan hesaplanabilir bir temsil olduğunu vurguladık. Geçersiz SMILES örneğinde `None` sonucunun neden önemli olduğunu gördük. Canonical SMILES'ın aynı molekülün farklı yazımlarını karşılaştırmada yararlı olabileceğini, ama tek başına bilimsel veri temizliği anlamına gelmediğini belirttik. Basit atom, bağ, formül ve molekül ağırlığı kontrollerinin RDKit nesnesini anlamaya yardım ettiğini gördük. Görselleştirmenin kalite kontrol düşüncesine destek olabileceğini konuştuk. Bütün bunlar, ileride ¹³C NMR projesinde yapı bilgisini daha dikkatli kullanmaya hazırlık içindir.


In [ ]:
notebook_boundary_check = {
    "uses_toy_molecules_only": True,
    "reads_nmredata_learning_sample": False,
    "downloads_data": False,
    "writes_files": False,
    "trains_model": False,
    "creates_train_test_split": False,
    "computes_metrics_or_benchmark": False,
    "selects_final_dataset": False,
    "accepts_adr": False,
}

for key, value in notebook_boundary_check.items():
    print(f"{key}: {value}")
